### Collect data

In [1]:
import pandas as pd
datacol = pd.read_csv('./data/benchmark_24-replicas.csv')
datacol.head()

,adapter,directory,path,replica_no,replicas,name,p,n,description,p_informative
0,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,1,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
1,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,2,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
2,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,3,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
3,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,4,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
4,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,5,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"


In [2]:
from tqdm import tqdm

rankdata = pd.DataFrame()
chunks = []
for _, dataset in tqdm(datacol.iterrows(), total=len(datacol)):
    rankpath = os.path.join('../../results/11714894_subsets/ranking', dataset.path + '.csv')
    rankpath2 = os.path.join('../../results/11824789/ranking', dataset.path + '.csv')

    data = pd.DataFrame()
    if os.path.exists(rankpath):
        data = pd.read_csv(rankpath)
    if os.path.exists(rankpath2):
        data = pd.read_csv(rankpath2)
    if not data.empty:
        dataset = dataset.drop(labels=[\
            'directory', 'path', 'adapter', 'p', 'n', 'replicas']) # drop some cols to save space
        # add more specific info to the ranking rows; e.g. replica_no, etc.
        rich_data = data.assign(**dataset)
        chunks.append(rich_data)
rankdata = pd.concat(chunks)
del chunks
# rankdata.to_csv('./data/rankdata.csv', index=False)
print('')
rankdata.shape

100%|██████████| 1828/1828 [00:28<00:00, 64.68it/s]



(193755, 9)

Compute additional statistics

In [43]:
# statdata = rankdata[rankdata['replica_no']=='1']
# print(statdata.shape)
statdata = rankdata.dropna(subset=['p_informative'])
print(statdata.shape)

(168512, 9)


In [44]:
import time
import numpy as np

s = time.time()
statdata['subsets'] = statdata['subset'].str.split(pat=',')
statdata['p_informatives'] = statdata['p_informative'].str.split(pat=',')
statdata['total_positives'] = statdata['p_informatives'].str.len()
statdata['n_positives'] = statdata['subsets'].str.len()
print('{} seconds'.format(time.time() - s))

8.497159957885742 seconds


In [45]:
tqdm.pandas()
def compute_tp_all(x):
    return np.sum(np.isin(x.subsets, x.p_informatives))
statdata['tp'] = statdata.progress_apply(compute_tp_all, axis='columns')

100%|██████████| 168512/168512 [00:44<00:00, 3806.00it/s]


In [47]:
s = time.time()
statdata['fp']          = statdata['n_positives'] - statdata['tp']
statdata['fn']          = statdata['total_positives'] - statdata['tp']
statdata['recall']      = statdata['tp'] / statdata['total_positives']
statdata['precision']   = statdata['tp'] / statdata['n_positives']
statdata['f1']          = 2 * ((statdata['precision'] * statdata['recall']) / \
                               (statdata['precision'] + statdata['recall']))
statdata.to_csv('./data/statdata.csv',\
    columns=[
        'description', 'ranking_method', 'replica_no', # identifiers
        'name', # semantics
        'cpu_time', 'feature_rank', # raw experiment data
        'recall', 'precision', 'f1' # computed metrics
    ], 
    index=False)
print('statdata successfully exported.')
print('{} seconds'.format(time.time() - s))

statdata successfully exported.
2.7959249019622803 seconds
